# 서울시 119 구급·구조 활동 데이터 전처리

응급실/구급 수요 예측 프로젝트를 위한 데이터 전처리 파이프라인입니다.

**공통 기준**
- 기준 키: `District`(서울시 25개 표준 자치구명), `ds`(ISO 날짜)
- 소방서명 등 지역 표기는 표준 자치구명으로 통일
- 구급활동 데이터는 출동 기록 없는 구간을 0으로 채움 (임의 대체분은 `imputed_zero` 플래그로 표시)
- 날짜 공백은 선형보간으로 처리
- 사고 원인 누락값은 `기타/미상`으로 대체 후 One-Hot Encoding

**출력 파일 (총 7개)**
| 파일 | 용도 | 스키마 |
|---|---|---|
| `ambulance_prophet.csv` | Prophet (장기추세) | ds, District, y, transport_cnt, transport_person, imputed_zero |
| `population_rf.csv` | Random Forest (인구밀집도) | ds, District, total_pop, Day_Pop, Night_Pop, Pop_Peak, Pop_Diff 등 |
| `accident_cause_long.csv` / `accident_cause_onehot.csv` | 사고 원인 통계 | ds, District, Cause, rescued_count (+ One-Hot) |
| `ambulance_monthly_gu_2022_2024.csv` | 최근 구단위 월별 보완 | ds, District, y |
| `ambulance_citywide_monthly.csv` | 전국API 기반 서울 전체 보완 | ds, District, y, transport_cnt, transport_person |
| `ambulance_sample_gwanak_202506.csv` | 관악구 샘플(참고용) | ds, District, y, note |


## 0. 환경 설정

In [8]:
import pandas as pd
import numpy as np
import re
import os


SRC_DIR = "C:/Users/Admin/Desktop/1차 프로젝트/uploads"
OUT_DIR = "C:/Users/Admin/Desktop/1차 프로젝트/outputs"
os.makedirs(OUT_DIR, exist_ok=True)

pd.set_option('display.max_columns', 20)


## 1. 소방서 → 자치구 매핑 준비

`서울소방서_119안전센터_현황.csv`의 관할구역 텍스트에서 자치구명을 추출해 소방서명 → 자치구 매핑 테이블을 만듭니다.
(전 25개 소방서가 1:1로 정확히 매핑되는지 검증했습니다.)

In [9]:
station_df = pd.read_csv(f"{SRC_DIR}/서울소방서_119안전센터_현황.csv", encoding="cp949")

def extract_gu(text):
    if pd.isna(text):
        return None
    m = re.search(r'([가-힣]+구)', str(text))
    return m.group(1) if m else None

station_df['District_guess'] = station_df['관할구역'].apply(extract_gu)
STATION2GU = station_df.groupby('서별')['District_guess'] \
    .agg(lambda x: x.value_counts().index[0]).to_dict()

VALID_GU = {'종로구','중구','용산구','성동구','광진구','동대문구','중랑구','성북구','강북구','도봉구',
            '노원구','은평구','서대문구','마포구','양천구','강서구','구로구','금천구','영등포구',
            '동작구','관악구','서초구','강남구','송파구','강동구'}

print(f"매핑된 소방서 수: {len(STATION2GU)} / 표준 자치구 수: {len(VALID_GU)}")
STATION2GU


FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/Admin/Desktop/1차 프로젝트/uploads/서울소방서_119안전센터_현황.csv'

## 2. 구급활동 실적 (Prophet용) — `ambulance_prophet.csv`

`119_구급활동_실적_소방서별__2026...csv` (2005~2025년, 소방서 단위)는 KOSIS 피벗 형식으로,
3개 헤더 행(연도 / 대분류 / 소분류)이 컬럼에 걸쳐 있습니다.
- Row0(컬럼명): 연도 (예: `2005`, `2005.1`, ...)
- Row1(iloc[0]): 대분류 (출동건수(건), 이송건수(건), 이송인원(명) ...)
- Row2(iloc[1]): 소분류 (소계, 급만성질환, 교통사고 ...)
- Row3(iloc[2:]): 실제 데이터 (소방서별 값)

여기서 `출동건수(건)-소계`를 Prophet의 `y`로, `이송건수`/`이송인원`은 보조 컬럼으로 추출합니다.

In [ ]:
SRC = f"{SRC_DIR}/119_구급활동_실적_소방서별__20260921151015.csv"

df = pd.read_csv(SRC, encoding="utf-8-sig")
level1 = df.iloc[0]
level2 = df.iloc[1]
data = df.iloc[2:].reset_index(drop=True)
cols = df.columns.tolist()
years = [c.split('.')[0] for c in cols]
id0, id1 = cols[0], cols[1]

records = []
for i, col in enumerate(cols):
    if col in (id0, id1) or not years[i].isdigit():
        continue
    yr = years[i]
    l1 = str(level1[col]).strip()
    l2 = str(level2[col]).strip()
    if l2 != '소계':
        continue
    if l1.startswith('출동건수'):
        metric = 'y'
    elif l1.startswith('이송건수'):
        metric = 'transport_cnt'
    elif l1.startswith('이송인원'):
        metric = 'transport_person'
    else:
        continue
    for _, row in data.iterrows():
        station = row[id1]
        if station not in STATION2GU:
            continue
        raw = row[col]
        is_missing = (str(raw).strip() == '-') or pd.isna(raw)
        try:
            val = float(str(raw).replace(',', '')) if not is_missing else 0.0
        except Exception:
            val = 0.0
            is_missing = True
        records.append({
            'ds': f'{yr}-01-01', 'District': STATION2GU[station],
            'metric': metric, 'value': val, 'imputed_zero': is_missing
        })

long_df = pd.DataFrame(records)
wide = long_df.pivot_table(index=['ds','District'], columns='metric',
                            values='value', aggfunc='first').reset_index()
flag = long_df[long_df['metric']=='y'][['ds','District','imputed_zero']]
ambulance_prophet = wide.merge(flag, on=['ds','District'], how='left')
ambulance_prophet = ambulance_prophet[['ds','District','y','transport_cnt','transport_person','imputed_zero']]
ambulance_prophet = ambulance_prophet.sort_values(['District','ds']).reset_index(drop=True)

print(ambulance_prophet.shape)
print("imputed_zero(0으로 대체된 결측) 건수:", ambulance_prophet['imputed_zero'].sum())
ambulance_prophet.to_csv(f"{OUT_DIR}/ambulance_prophet.csv", index=False, encoding='utf-8-sig')
ambulance_prophet.head()


## 3. 서울생활인구 (Random Forest용) — `population_rf.csv`

`자치구단위_서울생활인구_일별_집계표.csv`는 일단위 × 자치구 집계 데이터입니다.
- 자치구별로 날짜 인덱스를 재색인해 빠진 날짜(2019-10-15~27, 13일)를 찾아 **선형보간**
- `Day_Pop`(주간인구), `Night_Pop`(야간인구), `Pop_Peak`(일최대인구), `Pop_Diff`(주-야간 차이) 파생변수 생성

> 참고: 원본이 **일단위** 집계라 `hour`(시간대) 컬럼은 만들 수 없어 제외했습니다. 시간대별 데이터가 필요하면 별도 시간대 단위 원본이 있어야 합니다.

In [ ]:
SRC = f"{SRC_DIR}/자치구단위_서울생활인구_일별_집계표.csv"
pop = pd.read_csv(SRC, encoding="cp949")

pop['ds'] = pd.to_datetime(pop['기준일ID'].astype(str), format='%Y%m%d')
pop = pop[pop['시군구명'] != '서울시'].copy()   # 자치구 단위만 (서울시 합계행 제외)
pop = pop.rename(columns={
    '시군구명': 'District',
    '총생활인구수': 'total_pop',
    '주간인구수(09~18)': 'Day_Pop',
    '야간인구수(19~08)': 'Night_Pop',
})

keep_cols = ['ds','District','total_pop','내국인생활인구수','장기체류외국인인구수','단기체류외국인인구수',
             '일최대인구수','일최소인구수','Day_Pop','Night_Pop','일최대이동인구수',
             '서울외유입인구수','동일자치구행정동간이동인구수','자치구간이동인구수']
pop = pop[keep_cols]

full_dates = pd.date_range(pop['ds'].min(), pop['ds'].max(), freq='D')
filled = []
for gu, g in pop.groupby('District'):
    g = g.set_index('ds').sort_index().reindex(full_dates)
    g['District'] = gu
    num_cols = [c for c in keep_cols if c not in ('ds','District')]
    g[num_cols] = g[num_cols].interpolate(method='linear', limit_direction='both')
    g.index.name = 'ds'
    filled.append(g.reset_index())

population_rf = pd.concat(filled, ignore_index=True)
population_rf['Pop_Peak'] = population_rf['일최대인구수']
population_rf['Pop_Diff'] = population_rf['Day_Pop'] - population_rf['Night_Pop']
population_rf = population_rf.sort_values(['District','ds']).reset_index(drop=True)

print(population_rf.shape)
population_rf.to_csv(f"{OUT_DIR}/population_rf.csv", index=False, encoding='utf-8-sig')
population_rf.head()


## 4. 사고 원인 통계 — `accident_cause_long.csv` / `accident_cause_onehot.csv`

`119_구조활동_실적_구별__20260921153018.csv`(지표가 가장 상세한 버전)는 3개 헤더 행을 가진
KOSIS 피벗 형식입니다. `구조인원(명) > 사고종별구조인원 > {화재, 교통사고, ...}` 조합을 `Cause`로 추출합니다.
원인 카테고리명은 연도별로 표기가 달라 표준 명칭으로 통일하고, 정의되지 않은 값은 `기타/미상`으로 대체합니다.

In [ ]:
SRC = f"{SRC_DIR}/119_구조활동_실적_구별__20260921153018.csv"
df = pd.read_csv(SRC, encoding="utf-8-sig")

lvl1, lvl2, lvl3 = df.iloc[0], df.iloc[1], df.iloc[2]
data = df.iloc[3:].reset_index(drop=True)
cols = df.columns.tolist()
years = [c.split('.')[0] for c in cols]
id0, id1 = cols[0], cols[1]

records = []
for i, col in enumerate(cols):
    if col in (id0, id1) or not years[i].isdigit():
        continue
    yr = years[i]
    if str(lvl1[col]).strip() != '구조인원 (명)' or str(lvl2[col]).strip() != '사고종별구조인원':
        continue
    cause = str(lvl3[col]).strip()
    if cause == '소계':
        continue
    for _, row in data.iterrows():
        gu = row[id1]
        if gu not in VALID_GU:
            continue
        raw = row[col]
        missing = (str(raw).strip() in ('-', 'nan')) or pd.isna(raw)
        try:
            val = float(str(raw).replace(',', '')) if not missing else 0.0
        except Exception:
            val = 0.0
        records.append({'ds': f'{yr}-01-01', 'District': gu, 'Cause': cause, 'rescued_count': val})

cause_long = pd.DataFrame(records)

# 원인명 표준화 (연도별 지표 구성 차이 통합) + 결측/미정의 카테고리 -> 기타/미상
CAUSE_MAP = {
    '화재':'화재','교통사고':'교통사고','수난사고':'수난사고','기계사고':'기계사고',
    '건물사고':'건물사고','승강기사고':'승강기사고','산악사고':'산악사고','인명갇힘':'인명갇힘',
    '폭발사고':'폭발사고','잠금장치 개방':'잠금장치개방','안전조치':'안전조치',
    '자살추정':'자살추정','위치추적':'위치추적','기타':'기타/미상',
}
cause_long['Cause'] = cause_long['Cause'].map(CAUSE_MAP).fillna('기타/미상')

accident_cause_long = cause_long.groupby(['ds','District','Cause'], as_index=False)['rescued_count'].sum()
accident_cause_long.to_csv(f"{OUT_DIR}/accident_cause_long.csv", index=False, encoding='utf-8-sig')

# One-Hot Encoding (구·연도 단위로 원인별 건수를 컬럼화)
accident_cause_onehot = accident_cause_long.pivot_table(
    index=['ds','District'], columns='Cause', values='rescued_count',
    aggfunc='sum', fill_value=0).reset_index()
accident_cause_onehot.to_csv(f"{OUT_DIR}/accident_cause_onehot.csv", index=False, encoding='utf-8-sig')

print("long:", accident_cause_long.shape, " onehot:", accident_cause_onehot.shape)
print("Cause 목록:", sorted(accident_cause_long['Cause'].unique()))
accident_cause_long.head()


## 5. 보조 데이터 3종

기본 6개 출처 매핑 표에는 없지만 함께 업로드된 파일들을 가능한 범위에서 서울 구단위로 반영합니다.

1. **`ambulance_monthly_gu_2022_2024.csv`**: `서울시_소방_구급_출동_현황.xlsx`의 구/행정동 계층 구조에서
   구단위 행만 추출한 월별 시계열 (2022~2024년, 최근 구간 보완용)
2. **`ambulance_citywide_monthly.csv`**: `소방청_본부별_구급활동정보`(전국 시도본부 단위)에서 '서울' 행만 필터.
   이 데이터는 **시도 단위**라 25개 구로 분해할 수 없어 "서울시 전체" 단일 계열로만 제공합니다.
3. **`ambulance_sample_gwanak_202506.csv`**: `data.csv`는 관악구 2025-06 한 달치 1000행 샘플뿐이라
   단일 참고 포인트로 집계하고, API 페이지 제한으로 전수 여부가 불확실함을 `note`로 남겼습니다.

In [ ]:
# 5-1) 소방청 전국 데이터 -> 서울 전체 월별 보조 시계열
d2020 = pd.read_csv(f"{SRC_DIR}/소방청_본부별_구급활동정보_20201231.csv", encoding='cp949')
d2023 = pd.read_csv(f"{SRC_DIR}/소방청_본부별_구급활동정보_20231231.csv", encoding='cp949')
nat = pd.concat([d2020, d2023], ignore_index=True).drop_duplicates(subset=['년도','월','본부구분'])
seoul_city = nat[nat['본부구분'] == '서울'].copy()
seoul_city['ds'] = pd.to_datetime(seoul_city['년도'].astype(str) + '-' +
                                   seoul_city['월'].astype(str).str.zfill(2) + '-01')
seoul_city = seoul_city.rename(columns={'출동건수':'y','이송건수':'transport_cnt','이송환자수':'transport_person'})
seoul_city['District'] = '서울시_전체(구분류불가)'
ambulance_citywide_monthly = seoul_city[['ds','District','y','transport_cnt','transport_person']].sort_values('ds')
ambulance_citywide_monthly.to_csv(f"{OUT_DIR}/ambulance_citywide_monthly.csv", index=False, encoding='utf-8-sig')
print("서울 전체 월별:", ambulance_citywide_monthly.shape)

# 5-2) 서울시 소방 구급 출동 현황(xlsx) -> 구단위 월별 (2022-2024)
xl = pd.ExcelFile(f"{SRC_DIR}/서울시_소방_구급_출동_현황_2022_2024_.xlsx")
frames = []
for sheet in xl.sheet_names:
    year = re.search(r'\d{4}', sheet).group()
    d = xl.parse(sheet)
    d = d[d['구분'].isin(VALID_GU)].copy()
    d = d.groupby('구분', as_index=False).sum(numeric_only=True)
    long_ = d.melt(id_vars='구분', var_name='월', value_name='y')
    long_['월'] = long_['월'].str.replace('월', '').astype(int)
    long_['ds'] = pd.to_datetime(year + '-' + long_['월'].astype(str).str.zfill(2) + '-01')
    long_ = long_.rename(columns={'구분': 'District'})[['ds', 'District', 'y']]
    frames.append(long_)
ambulance_monthly_gu = pd.concat(frames, ignore_index=True).sort_values(['District', 'ds']).reset_index(drop=True)
ambulance_monthly_gu['y'] = ambulance_monthly_gu['y'].fillna(0)
ambulance_monthly_gu.to_csv(f"{OUT_DIR}/ambulance_monthly_gu_2022_2024.csv", index=False, encoding='utf-8-sig')
print("구단위 월별(2022-2024):", ambulance_monthly_gu.shape)

# 5-3) data.csv(관악소방서 샘플) -> 참고용 단일 포인트
sample = pd.read_csv(f"{SRC_DIR}/data.csv", encoding='utf-8-sig')
ambulance_sample_gwanak = pd.DataFrame([{
    'ds': '2025-06-01', 'District': '관악구', 'y': len(sample),
    'note': 'data.csv 원본 행수 기준 집계 (API 샘플 추정, 전수조사 여부 불확실 - 검증 필요)'
}])
ambulance_sample_gwanak.to_csv(f"{OUT_DIR}/ambulance_sample_gwanak_202506.csv", index=False, encoding='utf-8-sig')
print("관악구 샘플 포인트:", len(sample), "건")


## 6. 결과 요약

In [ ]:
summary = {
    'ambulance_prophet.csv': ambulance_prophet.shape,
    'population_rf.csv': population_rf.shape,
    'accident_cause_long.csv': accident_cause_long.shape,
    'accident_cause_onehot.csv': accident_cause_onehot.shape,
    'ambulance_citywide_monthly.csv': ambulance_citywide_monthly.shape,
    'ambulance_monthly_gu_2022_2024.csv': ambulance_monthly_gu.shape,
    'ambulance_sample_gwanak_202506.csv': ambulance_sample_gwanak.shape,
}
for k, v in summary.items():
    print(f"{k:45s} {v}")

print("\n※ 응급의료기관 정보 API / 실시간 응급실 병상 API는 정적 데이터가 아니라")
print("  실시간 호출이 필요한 부분으로, 이 노트북에는 포함되지 않았습니다.")
